<a href="https://colab.research.google.com/github/nevita275/ML_20_Nevita/blob/main/JS06/JS06-Tugas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TUGAS
# SOAL 1 - SVM pada voice.csv
## Persiapan: upload file

In [11]:
from google.colab import files
uploaded = files.upload()

Saving voice.csv to voice.csv


## Langkah 1 - Load data

In [12]:
import pandas as pd

df = pd.read_csv('voice.csv')
print(df.shape)
df.head()

(3168, 21)


,meanfreq,sd,median,Q25,Q75,IQR,skew,kurt,sp.ent,sfm,...,centroid,meanfun,minfun,maxfun,meandom,mindom,maxdom,dfrange,modindx,label
0,0.059781,0.064241,0.032027,0.015071,0.090193,0.075122,12.863462,274.402906,0.893369,0.491918,...,0.059781,0.084279,0.015702,0.275862,0.007812,0.007812,0.007812,0.000000,0.000000,male
1,0.066009,0.067310,0.040229,0.019414,0.092666,0.073252,22.423285,634.613855,0.892193,0.513724,...,0.066009,0.107937,0.015826,0.250000,0.009014,0.007812,0.054688,0.046875,0.052632,male
2,0.077316,0.083829,0.036718,0.008701,0.131908,0.123207,30.757155,1024.927705,0.846389,0.478905,...,0.077316,0.098706,0.015656,0.271186,0.007990,0.007812,0.015625,0.007812,0.046512,male
3,0.151228,0.072111,0.158011,0.096582,0.207955,0.111374,1.232831,4.177296,0.963322,0.727232,...,0.151228,0.088965,0.017798,0.250000,0.201497,0.007812,0.562500,0.554688,0.247119,male
4,0.135120,0.079146,0.124656,0.078720,0.206045,0.127325,1.101174,4.333713,0.971955,0.783568,...,0.135120,0.106398,0.016931,0.266667,0.712812,0.007812,5.484375,5.476562,0.208274,male


## Langkah 2 - Pisahkan fitur dan label

In [13]:
X = df.drop('label', axis=1)
y = df['label'].map({'male': 1, 'female': 0})

print(y.value_counts())

label
1    1584
0    1584
Name: count, dtype: int64


## Langkah 3 - Eksperimen 2 rasio × 3 kernel

In [14]:
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score

hasil = []

for nama_split, test_size in [('70:30', 0.3), ('80:20', 0.2)]:
    Xtr, Xte, ytr, yte = train_test_split(
        X, y, test_size=test_size, random_state=42, stratify=y)

    for kernel in ['linear', 'poly', 'rbf']:
        model = make_pipeline(StandardScaler(), SVC(kernel=kernel))
        model.fit(Xtr, ytr)
        acc = accuracy_score(yte, model.predict(Xte))
        hasil.append({'Split (train:test)': nama_split,
                      'Kernel': kernel,
                      'Akurasi': round(acc, 4)})

tabel = pd.DataFrame(hasil)
tabel

,Split (train:test),Kernel,Akurasi
0,70:30,linear,0.9790
1,70:30,poly,0.9600
2,70:30,rbf,0.9832
3,80:20,linear,0.9748
4,80:20,poly,0.9558
5,80:20,rbf,0.9826


## Langkah 4 - Tabel ringkas untuk laporan

In [15]:
tabel.pivot(index='Kernel', columns='Split (train:test)', values='Akurasi')

Split (train:test),70:30,80:20
Kernel,,
linear,0.9790,0.9748
poly,0.9600,0.9558
rbf,0.9832,0.9826


## Kesimpulan Soal 1:

* Kernel RBF paling akurat di kedua rasio (sekitar 0.983), lalu linear (0.975 sampai 0.979), lalu polynomial (0.956 sampai 0.960).
* Perbedaan antara 70:30 dan 80:20 kecil, jadi rasio tidak banyak mempengaruhi hasil.StandardScaler dipakai karena skala fitur sangat berbeda (misalnya kurt ribuan, meanfreq di bawah 1). Tanpa scaling, polynomial dan RBF anjlok (sekitar 0.51 dan 0.70 pada percobaan saya).
* stratify=y menjaga proporsi male/female seimbang di data latih dan uji.

<br> <br>

#SOAL 2 - SVM RBF dengan fitur histogram

## Langkah 1 - Ekstraksi fitur histogram

In [16]:
from google.colab import files
uploaded = files.upload()

Saving images.zip to images (1).zip


In [17]:
!unzip -qo images.zip
!ls images/training
!ls images/test

day  night
day  night


In [18]:
from pathlib import Path
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import cv2
import numpy as np
import pandas as pd

train_dir = "images/training/"
test_dir = "images/test/"

def load_dataset(img_dir):
    p = Path(img_dir)
    img_list = []
    for d in p.glob('*'):
        label = d.name
        for file in d.glob('*.jpg'):
            img = mpimg.imread(file)
            if img is not None:
                img_list.append((img, label))
    return img_list

def standarized_input(image):
    return cv2.resize(image, (1100, 600))

def label_encoder(label):
    return 1 if label == 'day' else 0

def preprocess(img_list):
    return [(standarized_input(img), label_encoder(lbl)) for img, lbl in img_list]

train_img = load_dataset(train_dir)
test_img = load_dataset(test_dir)
train_std_img_list = preprocess(train_img)
test_std_img_list = preprocess(test_img)

print(len(train_std_img_list), len(test_std_img_list))
print(train_std_img_list[0][0].shape)

240 160
(600, 1100, 3)


## Ekstraksi fitur histogram

In [19]:
def extract_histogram_feature(img_list, bins=32):
    feats, labels = [], []
    for img, label in img_list:
        # Ubah ke HSV, ambil kanal V (kecerahan)
        hsv = cv2.cvtColor(img, cv2.COLOR_RGB2HSV)
        hist = cv2.calcHist([hsv], [2], None, [bins], [0, 256]).flatten()
        # Normalisasi agar jumlah = 1
        hist = hist / hist.sum()
        feats.append(hist)
        labels.append(label)
    return np.array(feats), np.array(labels)

X_a, y_a = extract_histogram_feature(train_std_img_list)
X_b, y_b = extract_histogram_feature(test_std_img_list)
print(X_a.shape, X_b.shape)

(240, 32) (160, 32)


## Gabungkan lalu bagi ulang 80:20

In [20]:
from sklearn.model_selection import train_test_split

X = np.vstack([X_a, X_b])
y = np.concatenate([y_a, y_b])
print(X.shape, y.shape, np.bincount(y))

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
print(X_train.shape, X_test.shape)

(400, 32) (400,) [200 200]
(320, 32) (80, 32)


## Model dasar SVM RBF

In [21]:
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score

model_dasar = make_pipeline(StandardScaler(), SVC(kernel='rbf'))
model_dasar.fit(X_train, y_train)

acc_dasar = accuracy_score(y_test, model_dasar.predict(X_test))
print(f'Akurasi model dasar (C=1, gamma=scale): {acc_dasar:.4f}')

Akurasi model dasar (C=1, gamma=scale): 1.0000


## Hyperparameter tuning (GridSearchCV)

In [22]:
from sklearn.model_selection import GridSearchCV

param_grid = {'svc__C': [0.1, 1, 10, 100],
              'svc__gamma': [0.001, 0.01, 0.03, 0.1]}

grid = GridSearchCV(
    make_pipeline(StandardScaler(), SVC(kernel='rbf')),
    param_grid, cv=5, scoring='accuracy')
grid.fit(X_train, y_train)

print('Parameter terbaik :', grid.best_params_)
print(f'Akurasi CV terbaik: {grid.best_score_:.4f}')

Parameter terbaik : {'svc__C': 10, 'svc__gamma': 0.03}
Akurasi CV terbaik: 0.9875


## Evaluasi model terbaik pada data uji

In [23]:
from sklearn.metrics import classification_report, confusion_matrix

model_terbaik = grid.best_estimator_
y_pred = model_terbaik.predict(X_test)
acc_tuning = accuracy_score(y_test, y_pred)

print(f'Akurasi data uji setelah tuning: {acc_tuning:.4f}')
print(classification_report(y_test, y_pred, target_names=['night', 'day']))
print(confusion_matrix(y_test, y_pred))

Akurasi data uji setelah tuning: 1.0000
              precision    recall  f1-score   support

       night       1.00      1.00      1.00        40
         day       1.00      1.00      1.00        40

    accuracy                           1.00        80
   macro avg       1.00      1.00      1.00        80
weighted avg       1.00      1.00      1.00        80

[[40  0]
 [ 0 40]]


## Tabel hasil tuning

In [24]:
res = pd.DataFrame(grid.cv_results_)[
    ['param_svc__C', 'param_svc__gamma', 'mean_test_score']]
res.pivot(index='param_svc__C', columns='param_svc__gamma',
          values='mean_test_score').round(4)

param_svc__gamma,0.001,0.010,0.030,0.100
param_svc__C,,,,
0.1,0.8531,0.8875,0.9406,0.9500
1.0,0.9219,0.9531,0.9750,0.9812
10.0,0.9438,0.9781,0.9875,0.9875
100.0,0.9812,0.9844,0.9875,0.9875


## Catat performansi akhir

In [25]:
pd.DataFrame({
    'Model': ['SVM RBF dasar', 'SVM RBF setelah tuning'],
    'Akurasi (data uji 20%)': [round(acc_dasar, 4), round(acc_tuning, 4)]
})

,Model,Akurasi (data uji 20%)
0,SVM RBF dasar,1.0
1,SVM RBF setelah tuning,1.0


## Kesimpulan Soal 2

* SVM kernel RBF dengan fitur histogram kecerahan (32 bin pada kanal V ruang warna HSV) dan rasio 80:20 menghasilkan akurasi **1.0000** pada data uji, baik pada model dasar (C=1, gamma=scale) maupun setelah tuning.
* GridSearchCV memilih C=10 dan gamma=0.03 dengan akurasi cross-validation terbaik **0.9875** pada data latih.
* Tuning tidak mengubah akurasi data uji karena model dasar sudah mencapai akurasi maksimum. Dampak tuning terlihat pada tabel cross-validation: akurasi sangat bergantung pada C dan gamma, misalnya C=0.1 dengan gamma=0.001 hanya 0.8531, sedangkan C≥10 dengan gamma≥0.03 stabil di sekitar 0.9875.
* Fitur histogram efektif karena gambar siang didominasi piksel terang dan gambar malam didominasi piksel gelap, sehingga kedua kelas mudah dipisahkan.
* Catatan: data uji hanya 80 gambar, sehingga akurasi 1.0000 belum tentu mencerminkan performa pada gambar baru. Akurasi cross-validation (0.9875) memberi gambaran yang lebih konservatif.